In [ ]:
from brian2 import NeuronGroup, Synapses, SpikeMonitor, StateMonitor, run, ms, second, mV, mA
import matplotlib.pyplot as plt
import scipy.stats
import numpy as np

In [ ]:
def weight(theta1, theta2, basecase=False, shift=0):
    d = np.pi-np.abs(np.abs((theta1 + shift) - theta2)-np.pi)
    # w = np.maximum(scipy.stats.vonmises(loc=0,kappa=7).pdf(d) - scipy.stats.vonmises(loc=0,kappa=6).pdf(d), -.01)
    w = np.exp(-d**2*50) - .016*np.exp(-d**2/3)
    if basecase:
        return w
    return  w / weight(0, 0, basecase=True, shift=0)
    # return  d

theta_zero = np.pi
x = np.linspace(0,2*np.pi,200)
plt.plot(x, weight(theta_zero, x))
plt.axvline(x=theta_zero, color='k', linestyle='--')
plt.axhline(y=0, color='k', alpha=0.2)



In [ ]:

rng = np.random.default_rng(42)

N = 200
tau = 10*ms
sigma = .02*mV

# Add synaptic current to the neuron model
G = NeuronGroup(N, '''
dv/dt = (-v + I_syn + I_ext)/tau + sigma*sqrt(2/tau)*xi : volt  (unless refractory)
dI_syn/dt = -I_syn/(1*ms) : volt
I_ext : volt
theta : 1
                      ''',
                threshold='v>1*mV', reset='v=0*mV', refractory=1*ms)

G.theta = np.sort(rng.uniform(0, 2*np.pi, N))
G.v = 0
G.I_ext = 0*mV
G.I_syn = 0*mV


S = Synapses(G, G, 'w : volt', on_pre='I_syn_post += w')
S.connect()
# Set weights based on theta difference
for syn_idx in range(len(S)):
    i = S.i[syn_idx]
    j = S.j[syn_idx]

    # i = S.i[syn_idx]
    # j = S.j[syn_idx]
    # if i != j:
    #     S.w[syn_idx] = weight(G.theta[i], G.theta[j]) * 2*mV
    # else:
    #     S.w[syn_idx] = 0


    match (i - (j + 1)) % N:
        case 0:
            S.w[syn_idx] = 15*mV
        case 1 | -1:
            S.w[syn_idx] = 10*mV
        case 2 | -2:
            S.w[syn_idx] = 5*mV
        case 3 | -3:
            S.w[syn_idx] = 0*mV
        case 4 | -4:
            S.w[syn_idx] = -10*mV
        case _:
            S.w[syn_idx] = 0*mV


spike_monitor = SpikeMonitor(G, variables='v')
v_monitor = StateMonitor(G, variables='v', record=True)


run(100*ms)

r = .01
G.I_ext = ((np.pi-r < G.theta) & (G.theta < np.pi+r)) * 1.5*mV
run(20*ms)
G.I_ext = 0

run(10000*ms)



In [ ]:
-7 % 8

In [ ]:
%matplotlib inline
B = np.zeros((8,8))
for i in range(8):
    for j in range(8):
        if (i-(j+1)) % 8 == 1:
            B[i,j] = 1
plt.matshow(B)

In [ ]:
%matplotlib inline
plt.figure(figsize=(12, 6))

plt.subplot(2, 1, 1)
plt.plot(spike_monitor.t/ms, spike_monitor.i, '.k', markersize=2)
plt.xlabel('Time (ms)')
plt.ylabel('Neuron index')
plt.legend()
# plt.xlim(1500, 1550)
plt.ylim(0, N)



In [ ]:
# Plot membrane potential of a few neurons
%matplotlib inline
plt.figure(figsize=(12, 4))
for i in [48]:
    plt.plot(v_monitor.t/ms, v_monitor.v[i]/mV, alpha=0.7, label=f'Neuron {i}')
plt.xlabel('Time (ms)')
plt.ylabel('Membrane potential (mV)')
plt.title('Membrane Potential of Sample Neurons')
plt.axvline(x=100, color='r', linestyle='--', label='Kick')
plt.axvline(x=200, color='r', linestyle='--')
plt.axhline(y=0, color='k', alpha=.1)
plt.legend()
plt.xlim(0, 300)
plt.show()


In [ ]:
bins = np.arange(0, spike_monitor.t[-1]/ms, 10)
A = np.zeros((len(bins)-1, N))
for i in range(N):
    A[:,i] = np.histogram(spike_monitor.t[spike_monitor.i == i]/ms, bins=bins)[0]
from adaptive_latents import ArrayWithTime
A = ArrayWithTime(A, bins[1:])

In [ ]:
%matplotlib inline
plt.matshow(A)

In [ ]:
%matplotlib qt
from adaptive_latents import proSVD, sjPCA

pro_latents = proSVD(k=6, init_size=100).offline_run_on(A)

fig, ax = plt.subplots(subplot_kw=dict(projection='3d'))
plt.plot(pro_latents[:,0], pro_latents[:,2], pro_latents[:,1])


In [ ]:
%matplotlib inline
sjpca_latents = sjPCA(init_size=50).offline_run_on(pro_latents, show_tqdm=True)

from adaptive_latents.plotting_functions import AnimationManager, plot_history_with_tail

fig, axs = plt.subplots(ncols=4, nrows=1, figsize=(15, 5), width_ratios=[.2, .2, 1, 1], squeeze=False)
with AnimationManager(n_cols=3, dpi=100, fig=fig, fps=8, outdir='/tmp/', filename_stem='movie', filetype='mp4') as am:
    for t in np.arange(1500, 3000, 10):
        axs[0,0].cla()
        axs[0,0].plot(spike_monitor.t/ms, spike_monitor.i, '.k', markersize=2)
        axs[0,0].set_xlim(t-50, t)

        axs[0,1].cla()
        axs[0,1].matshow(A.slice_by_time(slice(t-50, t)).T, aspect='auto', origin='lower', vmin=A.min(), vmax=A.max())
        plot_history_with_tail(axs[0,2], data=pro_latents, current_t=t, tail_length=50, scatter_alpha=1)
        plot_history_with_tail(axs[0,3], data=sjpca_latents, current_t=t, tail_length=50, scatter_alpha=1)

        axs[0,0].set_title('Spike Raster')
        axs[0,1].set_title('Binned spikes')
        axs[0,2].set_title('proSVD Latents')
        axs[0,3].set_title('sjPCA Latents')
        am.grab_frame()

